In [1]:
!git clone -b dev-2 --single-branch https://github.com/Coftochka/CourseBDAMI3
!cp -r /kaggle/working/CourseBDAMI3/src /kaggle/working/
import sys
import os



Cloning into 'CourseBDAMI3'...
remote: Enumerating objects: 1245, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 1245 (delta 94), reused 190 (delta 72), pack-reused 1016 (from 3)
Receiving objects: 100% (1245/1245), 1.09 GiB | 24.71 MiB/s, done.
Resolving deltas: 100% (207/207), done.
Updating files: 100% (584/584), done.


In [2]:
sys.path.append('/kaggle/working/src')
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from data.dataloader import Dataloader
from data.tickers import TICKERS
from evaluation.metrics import regression_metrics, directional_accuracy
from evaluation.compare import ComparisonTable
from models import (
    LSTMModel, GRUModel, CNNModel, TransformerModel,
    LightGBMModel, ArimaModel,
)

245


In [3]:
FEATURE_COLS = [
    "open", "high", "low", "close", "volume",
    "sma5", "sma20", "ema12", "ema26", "close_sma20",
    "macd", "macd_signal", "macd_hist", "rsi14",
    "bb_pct", "bb_bw", "atr14", "obv",
]


loader = Dataloader(data_root="/kaggle/working/src/data/moex_candles")
dataset = loader.cut_on_windows(
    TICKERS,
    seq_len=60,
    step_size=1,
    feature_cols=FEATURE_COLS,
    interval="daily",
)
print(f"  Loaded {len(dataset)} / {len(TICKERS)} tickers")
X_train, y_train, X_val, y_val, X_test, y_test = loader.concat_splits(dataset)
print(f"  X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")


  Loaded 236 / 245 tickers
  X_train: (571508, 60, 18)  X_val: (98385, 60, 18)  X_test: (71247, 60, 18)


In [4]:
from models.run_optuna import *

In [10]:
m_lstm = LSTMModel.load("/kaggle/working/src/loaded_models/full_data_models/lstm_best.pth")
m_gru = GRUModel.load("/kaggle/working/src/loaded_models/full_data_models/gru_best.pth")
m_cnn = CNNModel.load("/kaggle/working/src/loaded_models/full_data_models/cnn_best.pth")
m_tr = TransformerModel.load("/kaggle/working/src/loaded_models/full_data_models/transformer_best.pth")
m_lgbm = LightGBMModel.load("/kaggle/working/src/loaded_models/full_data_models/lightgbm_best.pkl")


metrix_lstm = regression_metrics(y_test, m_lstm.predict(X_test), model_name="lstm")
metrix_gru = regression_metrics(y_test, m_gru.predict(X_test), model_name="gru")
metrix_cnn = regression_metrics(y_test, m_cnn.predict(X_test), model_name="cnn")
metrix_tr = regression_metrics(y_test, m_tr.predict(X_test), model_name="tr")
metrix_lgbm = regression_metrics(y_test, m_lgbm.predict(X_test), model_name="lgdm")


result = pd.concat([metrix_lstm, metrix_gru, metrix_cnn, metrix_tr, metrix_lgbm], ignore_index=True)
print(result)

        mae      rmse       mse          mape        r2  dir_accuracy  \
0  0.013500  0.021587  0.000466  8.892534e+05 -0.009703      0.491838   
1  0.013523  0.021544  0.000464  1.008333e+06 -0.005671      0.488470   
2  0.013325  0.021488  0.000462  3.704481e+04 -0.000455      0.465241   
3  0.013331  0.021485  0.000462  1.197644e+05 -0.000131      0.475122   
4  0.013328  0.021489  0.000462  1.402077e+05 -0.000528      0.477508   

         ic  n_samples  
0  0.043500      71247  
1  0.064438      71247  
2 -0.004307      71247  
3  0.010941      71247  
4  0.017231      71247  


# Бектестинг стратегии на одном тикере

Идея стратегии: модель предсказывает `log-return` следующего дня.  
- Если предсказание > threshold → **Long** (покупаем)  
- Если предсказание < -threshold → **Short** (продаём)  
- Иначе → **Out of market** (не торгуем)

Доходность считается как `signal * actual_log_return` по каждому окну тест-сета.

In [ ]:
# ── выбор портфеля из 10 тикеров и загрузка окон ────────────────────────────
PORTFOLIO_TICKERS = ["SBER", "GAZP", "LKOH", "GMKN", "ROSN",
                     "NVTK", "TATN", "MGNT", "MTSS", "YNDX"]

FEATURE_COLS = [
    "open", "high", "low", "close", "volume",
    "sma5", "sma20", "ema12", "ema26", "close_sma20",
    "macd", "macd_signal", "macd_hist", "rsi14",
    "bb_pct", "bb_bw", "atr14", "obv",
]

loader_bt = Dataloader(data_root="/kaggle/working/src/data/moex_candles")
dataset_bt = loader_bt.cut_on_windows(
    PORTFOLIO_TICKERS,
    seq_len=60,
    step_size=1,
    feature_cols=FEATURE_COLS,
    interval="daily",
)

# Оставляем только тикеры, для которых данные успешно загрузились
PORTFOLIO_TICKERS = [t for t in PORTFOLIO_TICKERS if t in dataset_bt]
print(f"Загружено тикеров: {len(PORTFOLIO_TICKERS)}: {PORTFOLIO_TICKERS}")

# Словари: ticker -> (X, y) для каждого сплита
splits = {t: loader_bt.splits_for_ticker(dataset_bt, t) for t in PORTFOLIO_TICKERS}
# splits[t] = (X_tr, y_tr, X_val, y_val, X_test, y_test)

for t in PORTFOLIO_TICKERS:
    X_tr, y_tr, X_val, y_val, X_test, y_test = splits[t]
    print(f"  {t}: train={X_tr.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

In [ ]:
# ── предсказания всех моделей для каждого тикера ─────────────────────────────
# preds_val[model_name][ticker] = np.ndarray (n_val,)
# preds_test[model_name][ticker] = np.ndarray (n_test,)

base_models = {
    "LSTM":        m_lstm,
    "GRU":         m_gru,
    "CNN":         m_cnn,
    "Transformer": m_tr,
    "LightGBM":    m_lgbm,
}

preds_val  = {name: {} for name in base_models}
preds_test = {name: {} for name in base_models}

for name, model in base_models.items():
    print(f"Predicting {name}...")
    for t in PORTFOLIO_TICKERS:
        X_tr, y_tr, X_val, y_val, X_test, y_test = splits[t]
        preds_val[name][t]  = model.predict(X_val)
        preds_test[name][t] = model.predict(X_test)

# ARIMA — fit per-ticker на train
print("Fitting ARIMA per ticker...")
preds_val["ARIMA"]  = {}
preds_test["ARIMA"] = {}
for t in PORTFOLIO_TICKERS:
    X_tr, y_tr, X_val, y_val, X_test, y_test = splits[t]
    m_arima_t = ArimaModel(p=1, d=0, q=1)
    m_arima_t.fit(X_tr, y_tr)
    preds_val["ARIMA"][t]  = m_arima_t.predict(X_val)
    preds_test["ARIMA"][t] = m_arima_t.predict(X_test)
    print(f"  {t} done")

all_model_names = list(preds_val.keys())
print(f"\nГотово. Модели: {all_model_names}")
print(f"Тикеры: {PORTFOLIO_TICKERS}")

In [ ]:
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression
# MS-AR добавляется в конце этой ячейки — после fit/predict для каждого тикера

class MarkovSwitchingARModel:
    """
    Per-window Markov Switching AutoRegression (MS-AR).

    Для каждого окна теста фитим MS-AR на значениях окна (первый признак —
    нормализованный close). One-step-ahead прогноз строится вручную:

        E[y_{t+1}] = sum_k P(s_t=k | data) * (const_k + phi_k * y_t)

    где P(s_t=k) берётся из smoothed_marginal_probabilities последнего шага,
    а параметры const_k, phi_k — из res.params.

    Параметры
    ----------
    k_regimes         : число скрытых режимов (2 или 3)
    order             : порядок AR в каждом режиме
    switching_variance: разные дисперсии по режимам
    """

    def __init__(self, k_regimes: int = 2, order: int = 1, switching_variance: bool = True):
        self.k_regimes = k_regimes
        self.order = order
        self.switching_variance = switching_variance
        self._fitted = False

    def fit(self, X: np.ndarray, y: np.ndarray, **kwargs) -> None:
        self._fitted = True   # параметры подбираются per-window в predict()

    def predict(self, X: np.ndarray) -> np.ndarray:
        assert self._fitted, "Call fit() first"
        series_all = np.asarray(X, dtype=np.float64)
        if series_all.ndim == 3:
            series_all = series_all[:, :, 0]   # (N, seq_len)

        preds_out = []
        for window in series_all:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    mod = MarkovAutoregression(
                        window,
                        k_regimes=self.k_regimes,
                        order=self.order,
                        switching_variance=self.switching_variance,
                    )
                    res = mod.fit(disp=False, maxiter=100)

                # res.predict() возвращает fitted values ряда (в масштабе window),
                # а не log-return. One-step-ahead прогноз следующего значения
                # считаем вручную: y_hat_{T+1} = sum_k P(s_T=k) * (const_k + phi_k * y_T)
                import math
                last_probs = np.array(res.smoothed_marginal_probabilities[-1])  # (k,)
                params = res.params  # плоский вектор всех параметров модели

                # Структура params для MS-AR(k=2, order=1, switching_variance=True):
                # [p00, p10,  const0, ar0_1, sigma0,  const1, ar1_1, sigma1]
                # Количество transition params = k*(k-1), затем k блоков [const, ar*order, sigma]
                n_transition = self.k_regimes * (self.k_regimes - 1)
                block_size = 1 + self.order + (1 if self.switching_variance else 0)
                y_last = float(window[-1])

                y_hat = 0.0
                for k in range(self.k_regimes):
                    offset = n_transition + k * block_size
                    const_k = float(params[offset])
                    ar_k    = [float(params[offset + 1 + j]) for j in range(self.order)]
                    ar_term = sum(ar_k[j] * float(window[-(j+1)]) for j in range(self.order))
                    y_hat  += float(last_probs[k]) * (const_k + ar_term)

                # log-return: предсказанное изменение нормализованного close
                log_ret = y_hat - y_last
                if math.isnan(log_ret) or math.isinf(log_ret):
                    log_ret = 0.0
                preds_out.append(log_ret)

            except Exception:
                preds_out.append(float(window[-1]))  # naive fallback

        result = np.array(preds_out, dtype=np.float32)
        # заменяем оставшиеся NaN/Inf наивным прогнозом (0.0)
        bad = ~np.isfinite(result)
        if bad.any():
            import warnings as _w
            _w.warn(f"MS-AR: {bad.sum()} NaN/Inf заменено на 0.0")
            result[bad] = 0.0
        return result


# Fit + predict MS-AR для каждого тикера
# Внимание: медленно (~3-5 мин на тикер) — EM-алгоритм фитится на каждом окне
preds_val["MS-AR"]  = {}
preds_test["MS-AR"] = {}

for t in PORTFOLIO_TICKERS:
    print(f"MS-AR fitting {t}...")
    X_tr, y_tr, X_val, y_val, X_test, y_test = splits[t]
    m_msar_t = MarkovSwitchingARModel(k_regimes=2, order=1, switching_variance=True)
    m_msar_t.fit(X_tr, y_tr)
    preds_val["MS-AR"][t]  = m_msar_t.predict(X_val)
    preds_test["MS-AR"][t] = m_msar_t.predict(X_test)
    print(f"  done: val={preds_val['MS-AR'][t].shape}  test={preds_test['MS-AR'][t].shape}")

all_model_names = list(preds_val.keys())
print(f"\nВсе модели готовы: {all_model_names}")

In [ ]:
# ── сборка матриц returns и предсказаний ─────────────────────────────────────
# Выравниваем длины test-окон по минимуму (у тикеров может отличаться на 1-2)
n_test_min = min(splits[t][5].shape[0] for t in PORTFOLIO_TICKERS)
n_val_min  = min(splits[t][3].shape[0] for t in PORTFOLIO_TICKERS)

# y_test_matrix: (n_tickers, n_test) — реальные log-returns на тесте
y_test_matrix = np.stack([splits[t][5][:n_test_min] for t in PORTFOLIO_TICKERS])  # (K, T)
y_val_matrix  = np.stack([splits[t][3][:n_val_min]  for t in PORTFOLIO_TICKERS])  # (K, T_val)

print(f"y_test_matrix: {y_test_matrix.shape}  (тикеры x окна)")
print(f"y_val_matrix:  {y_val_matrix.shape}")

# Ковариационная матрица из train-returns (для Марковица)
# Обрезаем по минимальной длине — у разных тикеров train может различаться
n_train_min = min(splits[t][1].shape[0] for t in PORTFOLIO_TICKERS)
y_train_matrix = np.stack([splits[t][1][:n_train_min] for t in PORTFOLIO_TICKERS])  # (K, T_train)
# Используем rolling-окно последних N_COV наблюдений train для оценки cov
N_COV = 252   # ~1 год торговых дней
cov_base = np.cov(y_train_matrix[:, -N_COV:])   # (K, K) — базовая ковариация
print(f"y_train_matrix: {y_train_matrix.shape}")
print(f"Ковариационная матрица: {cov_base.shape}, det={np.linalg.det(cov_base):.2e}")

In [ ]:
# ── Оптимизация Марковица ─────────────────────────────────────────────────────
from scipy.optimize import minimize

# Безрисковая ставка — ключевая ставка ЦБ РФ (среднее за 2024–2025 ~16%)
RF_ANNUAL = 0.16
RF_DAILY  = (1 + RF_ANNUAL) ** (1 / 252) - 1   # ≈ 0.000583 в день

def markowitz_weights(
    mu: np.ndarray,          # (K,) ожидаемые returns (предсказания модели)
    cov: np.ndarray,         # (K, K) ковариационная матрица
    risk_aversion: float = 1.0,   # lambda: баланс return/risk
    allow_short: bool = True,     # разрешить короткие позиции
    max_weight: float = 0.4,      # макс. вес одной бумаги
) -> np.ndarray:
    """
    Максимизируем: mu @ w - lambda/2 * w @ cov @ w
    при ограничениях: sum(w) = 1, |w_i| <= max_weight
    """
    K = len(mu)
    w0 = np.ones(K) / K

    def neg_utility(w):
        return -(mu @ w - risk_aversion / 2 * w @ cov @ w)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    lb = -max_weight if allow_short else 0.0
    bounds = [(lb, max_weight)] * K

    res = minimize(neg_utility, w0, method="SLSQP",
                   bounds=bounds, constraints=constraints,
                   options={"ftol": 1e-9, "maxiter": 500})
    if res.success:
        return res.x
    return w0   # fallback: равные веса


def portfolio_metrics(returns: np.ndarray, n_per_year: int = 252) -> dict:
    """Метрики портфеля по временному ряду дневных returns."""
    cum = np.expm1(np.cumsum(returns))
    peak = np.maximum.accumulate(cum + 1)
    dd = (cum + 1) / peak - 1
    excess = returns - RF_DAILY   # excess return над безрисковой ставкой
    ann_ret = float(np.expm1(returns.sum() * n_per_year / len(returns)))
    max_dd  = float(dd.min())
    calmar  = ann_ret / (abs(max_dd) + 1e-12)
    return {
        "total_return_%": round(float(np.expm1(returns.sum()) * 100), 2),
        "sharpe":         round(float(excess.mean() / (excess.std() + 1e-12) * np.sqrt(n_per_year)), 3),
        "max_drawdown_%": round(max_dd * 100, 2),
        "calmar":         round(calmar, 3),
        "win_rate_%":     round(float((returns > 0).mean() * 100), 2),
        "n_days":         len(returns),
    }

In [ ]:
# ── Портфельный бектест с оптимизацией Марковица ────────────────────────────
# Параметры
RISK_AVERSION  = 1.0    # lambda: чем больше, тем консервативнее
ALLOW_SHORT    = True   # разрешить короткие позиции
MAX_WEIGHT     = 0.4    # не более 40% в одну бумагу
TC             = 0.00015  # комиссия за сделку (0.015%)
REBALANCE_FREQ = 1        # ребалансировка каждые N дней (1 = ежедневно)

# Обновляем cov по скользящему окну: train + накопленный val
y_all_before_test = np.concatenate([y_train_matrix, y_val_matrix], axis=1)  # (K, T_tr+T_val)

portfolio_returns = {}   # model -> np.ndarray (n_test,)
weights_history   = {}   # model -> (n_test, K) матрица весов

for model_name in all_model_names:
    pred_dict = preds_test[model_name]
    n_steps = min(len(pred_dict[t]) for t in PORTFOLIO_TICKERS)

    port_ret   = np.zeros(n_steps)
    w_hist     = np.zeros((n_steps, len(PORTFOLIO_TICKERS)))
    prev_w     = np.ones(len(PORTFOLIO_TICKERS)) / len(PORTFOLIO_TICKERS)

    for i in range(n_steps):
        # Ожидаемые returns = предсказания модели на этом шаге
        mu = np.array([pred_dict[t][i] for t in PORTFOLIO_TICKERS])

        # Ребалансировка по расписанию
        if i % REBALANCE_FREQ == 0:
            # Скользящая ковариация: train + val + уже пройденные тест-шаги
            hist_returns = np.concatenate(
                [y_all_before_test, y_test_matrix[:, :i+1]], axis=1
            )
            n_cov = min(N_COV, hist_returns.shape[1])
            cov = np.cov(hist_returns[:, -n_cov:]) + np.eye(len(PORTFOLIO_TICKERS)) * 1e-6
            w = markowitz_weights(mu, cov, RISK_AVERSION, ALLOW_SHORT, MAX_WEIGHT)
        else:
            w = prev_w

        # Реальная доходность портфеля на этом шаге
        actual_ret = y_test_matrix[:, i]
        raw_port_ret = float(w @ actual_ret)

        # Комиссия за изменение весов
        turnover = np.sum(np.abs(w - prev_w))
        cost = turnover * TC
        port_ret[i] = raw_port_ret - cost

        w_hist[i]  = w
        prev_w     = w.copy()

    portfolio_returns[model_name] = port_ret
    weights_history[model_name]   = w_hist
    m = portfolio_metrics(port_ret)
    print(f"{model_name:12s}  total={m['total_return_%']:+7.2f}%  sharpe={m['sharpe']:+.3f}  maxDD={m['max_drawdown_%']:.2f}%")

# Бенчмарк: равновзвешенный Buy & Hold
bh_ret   = y_test_matrix.mean(axis=0)   # (T,) среднее по тикерам
bh_m     = portfolio_metrics(bh_ret)
print(f"\n{'EqualWeight B&H':12s}  total={bh_m['total_return_%']:+7.2f}%  sharpe={bh_m['sharpe']:+.3f}  maxDD={bh_m['max_drawdown_%']:.2f}%")

In [ ]:
# ── График 1: кумулятивная доходность всех моделей + Buy&Hold ────────────────
fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10.colors

for i, (name, ret) in enumerate(portfolio_returns.items()):
    cum = np.expm1(np.cumsum(ret)) * 100
    ax.plot(cum, label=name, color=colors[i], linewidth=1.6)

bh_cum = np.expm1(np.cumsum(bh_ret)) * 100
ax.plot(bh_cum, label="EqualWeight B&H", color="black", linewidth=2.0, linestyle="--")

ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
tickers_str = ", ".join(PORTFOLIO_TICKERS)
ax.set_title(f"Кумулятивная доходность портфеля (Марковиц)\n{tickers_str}", fontsize=12)
ax.set_xlabel("Торговый день (тест-сет)")
ax.set_ylabel("Доходность, %")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Сводная таблица метрик
rows = []
for name, ret in portfolio_returns.items():
    m = portfolio_metrics(ret)
    m["model"] = name
    rows.append(m)
bh_row = portfolio_metrics(bh_ret)
bh_row["model"] = "EqualWeight B&H"
rows.append(bh_row)

metrics_df = pd.DataFrame(rows).set_index("model")
print(f"\nПортфель: {PORTFOLIO_TICKERS}")
print(f"risk_aversion={RISK_AVERSION}  allow_short={ALLOW_SHORT}  max_weight={MAX_WEIGHT}  rebalance={REBALANCE_FREQ}d\n")
metrics_df

In [ ]:
# ── График 2: детальный разбор лучшей модели ─────────────────────────────────
BEST_MODEL = metrics_df.drop("EqualWeight B&H")["sharpe"].idxmax()
best_ret   = portfolio_returns[BEST_MODEL]
best_w     = weights_history[BEST_MODEL]   # (T, K)

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True,
                         gridspec_kw={"height_ratios": [3, 2, 2]})

# --- кумулятивная доходность ---
cum_best = np.expm1(np.cumsum(best_ret)) * 100
cum_bh   = np.expm1(np.cumsum(bh_ret))  * 100
axes[0].plot(cum_best, color="steelblue", linewidth=1.8, label=f"{BEST_MODEL}")
axes[0].plot(cum_bh,   color="black",     linewidth=1.5, linestyle="--", label="EqualWeight B&H")
axes[0].axhline(0, color="gray", linewidth=0.7, linestyle=":")
axes[0].set_ylabel("Доходность, %")
axes[0].set_title(f"Лучшая модель по Sharpe: {BEST_MODEL}", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- дневная доходность портфеля ---
axes[1].bar(range(len(best_ret)), best_ret * 100,
            color=np.where(best_ret >= 0, "green", "red"), alpha=0.6, width=1.0)
axes[1].axhline(0, color="gray", linewidth=0.7)
axes[1].set_ylabel("Return, %")
axes[1].set_title("Дневная доходность портфеля")
axes[1].grid(True, alpha=0.3)

# --- веса по тикерам (stacked area) ---
for k, t in enumerate(PORTFOLIO_TICKERS):
    axes[2].plot(best_w[:, k], label=t, linewidth=1.0)
axes[2].axhline(0, color="gray", linewidth=0.7, linestyle=":")
axes[2].set_ylabel("Вес в портфеле")
axes[2].set_xlabel("Торговый день (тест-сет)")
axes[2].set_title("Динамика весов Марковица")
axes[2].legend(fontsize=7, ncol=5, loc="upper right")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nМетрики {BEST_MODEL}:")
print(metrics_df.loc[BEST_MODEL].to_string())

In [ ]:
# ── Чувствительность к risk_aversion ─────────────────────────────────────────
risk_aversions = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sens_rows = []

for ra in risk_aversions:
    for model_name in all_model_names:
        pred_dict = preds_test[model_name]
        n_steps = min(len(pred_dict[t]) for t in PORTFOLIO_TICKERS)
        port_ret = np.zeros(n_steps)
        prev_w = np.ones(len(PORTFOLIO_TICKERS)) / len(PORTFOLIO_TICKERS)

        for i in range(n_steps):
            mu = np.array([pred_dict[t][i] for t in PORTFOLIO_TICKERS])
            hist = np.concatenate([y_all_before_test, y_test_matrix[:, :i+1]], axis=1)
            n_cov = min(N_COV, hist.shape[1])
            cov = np.cov(hist[:, -n_cov:]) + np.eye(len(PORTFOLIO_TICKERS)) * 1e-6
            w = markowitz_weights(mu, cov, ra, ALLOW_SHORT, MAX_WEIGHT)
            port_ret[i] = float(w @ y_test_matrix[:, i]) - np.sum(np.abs(w - prev_w)) * TC
            prev_w = w.copy()

        m = portfolio_metrics(port_ret)
        m["model"] = model_name
        m["risk_aversion"] = ra
        sens_rows.append(m)

sens_df = pd.DataFrame(sens_rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, ylabel in zip(
    axes,
    ["total_return_%", "sharpe", "max_drawdown_%"],
    ["Total return, %", "Sharpe ratio", "Max drawdown, %"],
):
    for name in all_model_names:
        sub = sens_df[sens_df["model"] == name]
        ax.semilogx(sub["risk_aversion"], sub[metric], marker="o", label=name)
    ax.set_xlabel("Risk aversion (lambda, log scale)")
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Чувствительность портфеля к risk aversion", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import random

# ── Анализ влияния размера портфеля на метрики ───────────────────────────────
# Для каждого размера N и каждой модели генерируем 100 случайных подвыборок
# тикеров из PORTFOLIO_TICKERS, запускаем бектест Марковица и усредняем метрики.

PORTFOLIO_SIZES = [1, 5, 10, 20, 25, 50]
N_BOOTSTRAP     = 100
MODELS_NO_MSAR  = [m for m in all_model_names if m != "MS-AR"]

# Все тикеры, для которых есть данные (используем глобальный dataset_bt)
ALL_AVAILABLE = list(dataset_bt.keys())
print(f"Доступно тикеров для выборки: {len(ALL_AVAILABLE)}")

def run_portfolio_backtest(tickers_subset, model_name):
    """Запускает бектест Марковица для подмножества тикеров, возвращает метрики."""
    K = len(tickers_subset)
    n_tr  = min(splits[t][1].shape[0] for t in tickers_subset)
    n_val = min(splits[t][3].shape[0] for t in tickers_subset)
    n_tst = min(splits[t][5].shape[0] for t in tickers_subset)

    y_tr  = np.stack([splits[t][1][:n_tr]  for t in tickers_subset])
    y_val = np.stack([splits[t][3][:n_val] for t in tickers_subset])
    y_tst = np.stack([splits[t][5][:n_tst] for t in tickers_subset])
    y_hist = np.concatenate([y_tr, y_val], axis=1)

    pred_dict = preds_test[model_name]
    n_steps = min(len(pred_dict[t]) for t in tickers_subset)
    n_steps = min(n_steps, n_tst)

    port_ret = np.zeros(n_steps)
    prev_w   = np.ones(K) / K

    for i in range(n_steps):
        mu  = np.array([pred_dict[t][i] for t in tickers_subset])
        hist = np.concatenate([y_hist, y_tst[:, :i+1]], axis=1)
        nc   = min(N_COV, hist.shape[1])
        cov  = np.cov(hist[:, -nc:]) if K > 1 else np.array([[hist[0, -nc:].var()]])
        cov  = np.atleast_2d(cov) + np.eye(K) * 1e-6
        w    = markowitz_weights(mu, cov, RISK_AVERSION, ALLOW_SHORT, MAX_WEIGHT)
        port_ret[i] = float(w @ y_tst[:, i]) - np.sum(np.abs(w - prev_w)) * TC
        prev_w = w.copy()

    # Calmar = annualised return / |max drawdown|
    cum = np.expm1(np.cumsum(port_ret))
    peak = np.maximum.accumulate(cum + 1)
    dd   = (cum + 1) / peak - 1
    ann_ret = float(np.expm1(port_ret.sum() * 252 / len(port_ret)))
    max_dd  = float(dd.min())
    calmar  = ann_ret / (abs(max_dd) + 1e-12)

    excess = port_ret - RF_DAILY
    return {
        "sharpe":         float(excess.mean() / (excess.std() + 1e-12) * np.sqrt(252)),
        "max_drawdown_%": max_dd * 100,
        "calmar":         calmar,
        "total_return_%": float(np.expm1(port_ret.sum()) * 100),
        "win_rate_%":     float((port_ret > 0).mean() * 100),
    }


# ── Основной цикл ─────────────────────────────────────────────────────────────
size_results = []   # list of dicts

for size in PORTFOLIO_SIZES:
    print(f"\n--- Размер портфеля: {size} ---")
    pool = ALL_AVAILABLE if size <= len(ALL_AVAILABLE) else ALL_AVAILABLE

    for model_name in MODELS_NO_MSAR:
        metrics_accum = []
        for _ in range(N_BOOTSTRAP):
            sample = random.sample(pool, min(size, len(pool)))
            # убеждаемся что для всех тикеров выборки есть предсказания
            sample = [t for t in sample if t in preds_test[model_name]]
            if len(sample) < 1:
                continue
            try:
                m = run_portfolio_backtest(sample, model_name)
                metrics_accum.append(m)
            except Exception:
                continue

        if not metrics_accum:
            continue

        avg = {k: float(np.mean([x[k] for x in metrics_accum])) for k in metrics_accum[0]}
        std = {k: float(np.std( [x[k] for x in metrics_accum])) for k in metrics_accum[0]}
        row = {"model": model_name, "portfolio_size": size, "n_samples": len(metrics_accum)}
        for k, v in avg.items():
            row[k]            = round(v, 4)
            row[f"{k}_std"]   = round(std[k], 4)
        size_results.append(row)
        print(f"  {model_name:12s}  sharpe={avg['sharpe']:+.3f}  calmar={avg['calmar']:+.3f}"
              f"  total={avg['total_return_%']:+.2f}%  maxDD={avg['max_drawdown_%']:.2f}%"
              f"  winrate={avg['win_rate_%']:.1f}%  (n={len(metrics_accum)})")

size_df = pd.DataFrame(size_results)
print("\n\nГотово.")
size_df

In [ ]:
# ── Графики: метрики vs размер портфеля ──────────────────────────────────────
metrics_to_plot = [
    ("sharpe",          "Sharpe ratio"),
    ("max_drawdown_%",  "Max Drawdown, %"),
    ("calmar",          "Calmar ratio"),
    ("total_return_%",  "Total Return, %"),
    ("win_rate_%",      "Win Rate, %"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors = plt.cm.tab10.colors

for ax, (metric, title) in zip(axes, metrics_to_plot):
    for i, model_name in enumerate(MODELS_NO_MSAR):
        sub = size_df[size_df["model"] == model_name].sort_values("portfolio_size")
        if sub.empty:
            continue
        ax.plot(sub["portfolio_size"], sub[metric],
                marker="o", label=model_name, color=colors[i], linewidth=1.8)
        ax.fill_between(
            sub["portfolio_size"],
            sub[metric] - sub[f"{metric}_std"],
            sub[metric] + sub[f"{metric}_std"],
            alpha=0.12, color=colors[i]
        )
    ax.set_xlabel("Размер портфеля (кол-во акций)")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticks(PORTFOLIO_SIZES)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Скрываем лишнюю ячейку (5 метрик, 6 subplots)
axes[-1].set_visible(False)

plt.suptitle("Влияние размера портфеля на метрики стратегии\n(среднее ± std по 100 bootstrap-выборкам)",
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Сводная таблица: среднее по bootstrap для каждой модели и размера ────────
pivot_metrics = ["sharpe", "calmar", "total_return_%", "max_drawdown_%", "win_rate_%"]

summary_frames = []
for metric in pivot_metrics:
    pivot = size_df.pivot(index="portfolio_size", columns="model", values=metric)
    pivot.columns = [f"{col}" for col in pivot.columns]
    pivot.index.name = "portfolio_size"
    pivot["metric"] = metric
    summary_frames.append(pivot.reset_index().set_index(["metric", "portfolio_size"]))

summary_table = pd.concat(summary_frames).round(4)

print("Сводная таблица (среднее по 100 bootstrap-выборкам):")
print(f"Строки: (метрика, размер портфеля)   Столбцы: модели\n")
print(summary_table.to_string())

# Красивый heatmap для Sharpe
sharpe_pivot = size_df.pivot(index="portfolio_size", columns="model", values="sharpe").round(3)

fig, ax = plt.subplots(figsize=(len(MODELS_NO_MSAR) * 1.4 + 1, 4))
im = ax.imshow(sharpe_pivot.values, aspect="auto", cmap="RdYlGn")
plt.colorbar(im, ax=ax, label="Sharpe ratio")
ax.set_xticks(range(len(sharpe_pivot.columns)))
ax.set_xticklabels(sharpe_pivot.columns, rotation=30, ha="right", fontsize=10)
ax.set_yticks(range(len(sharpe_pivot.index)))
ax.set_yticklabels(sharpe_pivot.index, fontsize=10)
ax.set_xlabel("Модель")
ax.set_ylabel("Размер портфеля")
ax.set_title("Heatmap: Sharpe ratio (среднее по bootstrap)")
for r in range(len(sharpe_pivot.index)):
    for c in range(len(sharpe_pivot.columns)):
        ax.text(c, r, f"{sharpe_pivot.values[r, c]:.3f}",
                ha="center", va="center", fontsize=9,
                color="black")
plt.tight_layout()
plt.show()